## Import

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

## Env

In [3]:
load_dotenv(override=True)

True

## Parameters

In [14]:
RUNS = 3

MODELS = [
    "openai/gpt-5.4-nano",
    "openai/gpt-5.4-mini",
    "openai/gpt-5.4",
#    "anthropic/claude-sonnet-4.5",
#    "google/gemini-2.5-flash",
#    "meta-llama/llama-3.3-70b-instruct",
#    "qwen/qwen3.6-plus",
#    "mistralai/mistral-large-2512",
]

# Claude:
# - anthropic/claude-sonnet-5
# - anthropic/claude-haiku-4.5

# Gemini:
# - google/gemini-3.1-pro-preview
# - google/gemini-3.5-flash
# - google/gemini-3.1-flash-lite-preview

# Gemma:
# - google/gemma-4-31b-it:free
# - google/gemma-3-27b-it

# Llama:
# - meta-llama/llama-4-maverick
# - meta-llama/llama-4-scout
# - meta-llama/llama-3.3-70b-instruct

# Qwen:
# - qwen/qwen3.7-max
# - qwen/qwen3.7-plus
# - qwen/qwen3.6-35b-a3b

# Mistral:
# - mistralai/mistral-large-2512
# - mistralai/mistral-small-2603
# - mistralai/ministral-14b-2512

TEMPERATURE = 0

SYSTEM_PROMPT = """
Person A and Person B are talking to each other. Person A asks a question and Person B answers.
Given Person A's question, estimate how strongly Person B's answer should be interpreted as
answering “yes” versus “no.”

Output a single integer score from 1 to 7, where:

1 = definitely no
2 = very likely no
3 = somewhat likely no
4 = completely uncertain, ambiguous
5 = somewhat likely yes
6 = very likely yes
7 = definitely yes

Also report the rationale for your decision in a few sentences, explaining why you gave the score you did.
If the answer is ambiguous or irrelevant, explain why.

Return score and rationale as valid JSON.
"""

SCORE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "score": {
                        "type": "integer"
                    },
                    "rationale": {"type": "string"}
                },
                "required": ["id", "score", "rationale"],
                "additionalProperties": False
            }
        }
    },
    "required": ["results"],
    "additionalProperties": False
}


##  Coe

In [13]:


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def read_qas(file_path: str, retain=None) -> list:
    """
    Read the data from the given CSV file and transform it into a json structure that can be used as batch input to the LLMs.

    Args:
        file_path (str): The path to the CSV file.
        retain (int): The number of rows to include in the output for direct and indirect qaüpairs respectively.
    """
    # read the csv file
    df = pd.read_csv(file_path, sep=";")

    # clip if necessary
    if retain is not None:
        if retain > len(df):
            raise ValueError(f"Retain value {retain} is greater than the number of qas in the dataframe {len(df)}.")
        df = df.head(retain)


    # extract only relevant information
    df_new = df[["CODE", "CONTEXT QUESTION", "CRITICAL UTTERANCE"]]
    df_new["text"]  = "Person A: " + df_new["CONTEXT QUESTION"] + "; Person B: " + df_new["CRITICAL UTTERANCE"]
    df_new.rename(columns={"CODE": "id"}, inplace=True)

    # convert into json
    output = [
    {
        "id": str(row["id"]),
        "text": row["text"]
    }
    for _, row in df_new.iterrows()
    ]

    return output



def score_qas(model: str, qas: list) -> dict:
    """
    Score the QAs using one OpenRouter model.
    """

    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": json.dumps(qas),
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "batch_classification_scores",
                "strict": True,
                "schema": SCORE_SCHEMA,
            },
        },
        extra_body={
            "provider": {
                "require_parameters": False,
                "allow_fallbacks": False,
            }
        },
    )

    content = response.choices[0].message.content
    result = json.loads(content)

    if isinstance(result, list):
        result = {"results": result}
    elif not isinstance(result, dict) or "results" not in result:
        raise ValueError(f"Unexpected response shape: {result!r}")

    for item in result["results"]:
        score = item["score"]
        if not isinstance(score, int) or not 1 <= score <= 7:
            raise ValueError(f"Invalid score {score!r} for item {item['id']}. Expected an integer from 1 to 7.")

    return result



def save_result(model: str, run: int, result):
    '''Save the scoring results to a CSV file.'''
    
    # Convert results to a dataframe
    results_df = pd.DataFrame(result["results"])

    # Save to CSV
    results_df.to_csv(f"data/raw/{model.replace("/", "_")}_score_run{run+1}.csv", index=False)


In [6]:
qas = read_qas("data/external/iCO-Eval2_summarizedRatings.csv", retain=5)

print(len(qas))



5


In [15]:
for model in MODELS:
    
    for run in tqdm(range(RUNS), desc=f"Currently collecting data from {model}"):
        result = score_qas(model, qas)
        save_result(model, run, result)




Currently collecting data from openai/gpt-5.4-nano: 100%|██████████| 3/3 [00:07<00:00,  2.62s/it]
Currently collecting data from openai/gpt-5.4-mini: 100%|██████████| 3/3 [00:05<00:00,  1.98s/it]
Currently collecting data from openai/gpt-5.4: 100%|██████████| 3/3 [00:09<00:00,  3.07s/it]
